##### ARTI 560 - Computer Vision

## Action Recognition - Exercise

### Objective

In this exercise, you will train a deep learning model to recognize three specific human actions using the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) and validate the model's real-world performance using external video data.

*[Note: This notebook is based on [this](https://github.com/Sumaya2026/learnopencv/tree/master/Optical-Flow-Estimation-using-Deep-Learning-RAFT) GitHub Repository by LearnOpenCV]*


#### Tasks

- Choose **three classes** from the UCF11 dataset (e.g., Basketball Shooting, Biking, Tennis Swinging, etc.).
- Preprocess the dataset.
- Split the data into training and testing.
- Create and train the model.
- Save the trained model .
    **Important Note**: The final trained model must be saved with a filename that includes your name. This is a mandatory step for the submission.
    ```
    # Example Code
    student_name = "YourName" # Replace with your actual name
    save_path = f"{student_name}_ucf11_model.h5"
    model.save(save_path)
    print(f"Model saved as {save_path}")
    ```
- Validate the model on 3 Youtube videos, each clearly showing one of your three chosen action classes.


In [1]:
import os
import cv2
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import TimeDistributed, Conv2D, MaxPooling2D, Flatten, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

In [2]:
seed_constant = 23
np.random.seed(seed_constant)
random.seed(seed_constant)
tf.random.set_seed(seed_constant)

In [3]:
!curl -k -L https://www.crcv.ucf.edu/data/UCF11_updated_mpg.rar -o UCF11_updated_mpg.rar
!unrar x UCF11_updated_mpg.rar -inul -y

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  996M  100  996M    0     0  12.6M      0  0:01:18  0:01:18 --:--:-- 24.8M


In [4]:
!ls -lh

total 997M
drwxr-xr-x  1 root root 4.0K Apr 16 13:33 sample_data
drwxr-xr-x 13 root root 4.0K Oct 19  2011 UCF11_updated_mpg
-rw-r--r--  1 root root 997M Apr 21 00:22 UCF11_updated_mpg.rar


In [16]:
image_height, image_width = 64, 64
sequence_length = 10
selected_classes = ['basketball', 'tennis', 'walking']
dataset_directory = 'UCF11_updated_mpg'

In [17]:
def find_class_directories(dataset_directory, selected_classes):
    class_directories = {}
    for item in sorted(os.listdir(dataset_directory)):
        item_path = os.path.join(dataset_directory, item)
        if not os.path.isdir(item_path):
            continue
        item_lower = item.lower().replace('_', ' ').replace('-', ' ')
        for class_name in selected_classes:
            if class_name in item_lower and class_name not in class_directories:
                class_directories[class_name] = item_path
    return class_directories


def frames_extraction(video_path, sequence_length, image_height, image_width):
    frames_list = []
    video_reader = cv2.VideoCapture(video_path)
    video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))

    if video_frames_count < sequence_length:
        video_reader.release()
        return frames_list

    skip_frames_window = max(int(video_frames_count / sequence_length), 1)

    for frame_counter in range(sequence_length):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)
        success, frame = video_reader.read()
        if not success:
            break

        resized_frame = cv2.resize(frame, (image_width, image_height))
        normalized_frame = resized_frame / 255.0
        frames_list.append(normalized_frame)

    video_reader.release()
    return frames_list


def create_dataset(dataset_directory, selected_classes, sequence_length, image_height, image_width):
    features = []
    labels = []
    video_paths = []

    class_directories = find_class_directories(dataset_directory, selected_classes)
    class_names = list(class_directories.keys())

    max_videos_per_class = 100

def create_dataset(dataset_directory, selected_classes, sequence_length, image_height, image_width):
    features = []
    labels = []
    video_paths = []

    class_directories = find_class_directories(dataset_directory, selected_classes)
    class_names = list(class_directories.keys())

    max_videos_per_class = 100

    for class_index, class_name in enumerate(class_names):
        print(f'Extracting data of class: {class_name}')
        class_directory = class_directories[class_name]

        count = 0

        for root, _, files in os.walk(class_directory):
            for file_name in files:

                if count >= max_videos_per_class:
                    break

                if file_name.endswith(('.avi', '.mpg', '.mpeg', '.mp4')):
                    video_file_path = os.path.join(root, file_name)

                    frames = frames_extraction(
                        video_file_path,
                        sequence_length,
                        image_height,
                        image_width
                    )

                    if len(frames) == sequence_length:
                        features.append(frames)
                        labels.append(class_index)
                        video_paths.append(video_file_path)

                        count += 1

            if count >= max_videos_per_class:
                break

    features = np.asarray(features)
    labels = np.array(labels)
    return features, labels, video_paths, class_names

In [18]:
features, labels, video_paths, class_names = create_dataset(
    dataset_directory,
    selected_classes,
    sequence_length,
    image_height,
    image_width
)

one_hot_encoded_labels = to_categorical(labels)

features_train, features_test, labels_train, labels_test = train_test_split(
    features,
    one_hot_encoded_labels,
    test_size=0.2,
    shuffle=True,
    random_state=seed_constant,
    stratify=labels
)

print(features_train.shape, features_test.shape)
print(class_names)

Extracting data of class: basketball
Extracting data of class: tennis
Extracting data of class: walking
(240, 10, 64, 64, 3) (60, 10, 64, 64, 3)
['basketball', 'tennis', 'walking']


In [19]:
def create_model(sequence_length, image_height, image_width, classes_count):
    model = Sequential()
    model.add(TimeDistributed(Conv2D(16, (3, 3), activation='relu', padding='same'),
                              input_shape=(sequence_length, image_height, image_width, 3)))
    model.add(TimeDistributed(MaxPooling2D((2, 2))))
    model.add(TimeDistributed(Conv2D(32, (3, 3), activation='relu', padding='same')))
    model.add(TimeDistributed(MaxPooling2D((2, 2))))
    model.add(TimeDistributed(Flatten()))
    model.add(LSTM(64))
    model.add(Dropout(0.5))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(classes_count, activation='softmax'))
    return model

model = create_model(sequence_length, image_height, image_width, len(class_names))
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed_5              │ (None, 10, 64, 64, 16) │           448 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_6              │ (None, 10, 32, 32, 16) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_7              │ (None, 10, 32, 32, 32) │         4,640 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_8              │ (None, 10, 16, 16, 32) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_9              │ (None, 10, 8192)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │     2,113,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,121,059 (8.09 MB)

 Trainable params: 2,121,059 (8.09 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
model.compile(loss='categorical_crossentropy', optimizer='Adam', metrics=['accuracy'])

early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

model_training_history = model.fit(
    x=features_train,
    y=labels_train,
    epochs=30,
    batch_size=8,
    shuffle=True,
    validation_split=0.2,
    callbacks=[early_stopping_callback]
)

Epoch 1/30
24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 57ms/step - accuracy: 0.3177 - loss: 1.1490 - val_accuracy: 0.4583 - val_loss: 1.0989
Epoch 2/30
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.5260 - loss: 1.0485 - val_accuracy: 0.4583 - val_loss: 1.0341
Epoch 3/30
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.6250 - loss: 0.8929 - val_accuracy: 0.5000 - val_loss: 0.9536
Epoch 4/30
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.7396 - loss: 0.6649 - val_accuracy: 0.7708 - val_loss: 0.7298
Epoch 5/30
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.8021 - loss: 0.5675 - val_accuracy: 0.7708 - val_loss: 0.5329
Epoch 6/30
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.9219 - loss: 0.2707 - val_accuracy: 0.8542 - val_loss: 0.3736
Epoch 7/30
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.9427 - loss: 0.1875 - val_accuracy: 0.8542 - val_loss: 0.4656
Epoch 8/30
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.9740 - loss: 0.0990 - val_accuracy: 0.8542 - v

In [21]:
model_evaluation_history = model.evaluate(features_test, labels_test)
print('Test Loss:', model_evaluation_history[0])
print('Test Accuracy:', model_evaluation_history[1])

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step - accuracy: 0.8667 - loss: 0.3104
Test Loss: 0.31041771173477173
Test Accuracy: 0.8666666746139526


In [22]:
student_name = "NadaAlghamdi"
save_path = f"{student_name}_ucf11_model.h5"
model.save(save_path)
print(f"Model saved as {save_path}")

Model saved as NadaAlghamdi_ucf11_model.h5


In [ ]:
#!pip install -q yt-dlp
import yt_dlp

In [24]:
def download_youtube_video(video_url, output_path):
    ydl_opts = {
        'format': 'mp4/bestvideo+bestaudio/best',
        'outtmpl': output_path,
        'quiet': True,
        'merge_output_format': 'mp4'
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([video_url])


def predict_video_class(video_path, model, sequence_length, image_height, image_width, class_names):
    frames = frames_extraction(video_path, sequence_length, image_height, image_width)
    if len(frames) != sequence_length:
        return None
    predicted_probabilities = model.predict(np.expand_dims(frames, axis=0), verbose=0)[0]
    predicted_label_index = np.argmax(predicted_probabilities)
    return class_names[predicted_label_index], predicted_probabilities[predicted_label_index]

In [25]:
youtube_videos = {
    'basketball': 'https://www.youtube.com/watch?v=SyvuSxCyfi0',
    'tennis': 'https://www.youtube.com/watch?v=se3MGnTv2eY',
    'walking': 'https://www.youtube.com/watch?v=Mol0lrRBy3g'
}

os.makedirs('youtube_validation', exist_ok=True)

for expected_class, video_url in youtube_videos.items():
    video_file_path = os.path.join('youtube_validation', f'{expected_class}.mp4')
    download_youtube_video(video_url, video_file_path)
    prediction = predict_video_class(
        video_file_path,
        model,
        sequence_length,
        image_height,
        image_width,
        class_names
    )
    print(expected_class, prediction)

basketball ('basketball', np.float32(0.49803832))


tennis ('basketball', np.float32(0.82994765))


walking ('walking', np.float32(0.85382164))
